In [18]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv(override=True)

gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY is not set. Add it to your .env file.")

genai.configure(api_key=gemini_api_key)
gemini_client = genai.GenerativeModel("gemini-2.5-flash")

print("API client successfully configured")
print(f"Loaded GEMINI_API_KEY: {gemini_api_key[:5]}...")

API client successfully configured
Loaded GEMINI_API_KEY: AIzaS...


In [19]:
from IPython.display import display, Markdown

def print_markdown(text):
    display(Markdown(text))

In [ ]:
def get_ai_tutor_response(user_question):
    """
    Sends a question to the Gemini API, asking it to respond as an AI Tutor.
    Args:
        user_question (str): The question asked by the user
    Returns:
        str: The AI's response, or an error message
    """
    system_prompt = "You are a helpful and patient AI Tutor. Explain concepts clearly and concisely."

    try:
        response = gemini_client.generate_content(
            f"{system_prompt}\n\nStudent question: {user_question}",
            generation_config=genai.types.GenerationConfig(
                temperature=0.7
            )
        )
        return response.text
    except Exception as e:
        return f"Sorry, I encountered an error trying to get an answer: {e}"


In [21]:
test_question = "Could you explain the concept of functions in Python and their purpose in programming?"

tutor_answer = get_ai_tutor_response(test_question)

print_markdown(f"""
### Asking the AI Tutor

{test_question}

### AI Tutor's Response

{tutor_answer}
""")


### Asking the AI Tutor

Could you explain the concept of functions in Python and their purpose in programming?

### AI Tutor's Response

Hello! I'd be happy to explain functions in Python.

---

### What is a Function in Python?

Imagine you have a specific task that you need to perform multiple times in your program, like calculating the area of a circle or greeting a user. Instead of writing the same lines of code over and over, you can bundle them up into a reusable unit called a **function**.

Think of a function as a mini-program or a recipe:

*   It has a **name** (like "bake a cake").
*   It takes specific **ingredients** or inputs (like flour, eggs, sugar).
*   It performs a series of **steps** (mixing, baking, frosting).
*   It produces an **output** (a delicious cake!).

In Python, a function is a block of organized, reusable code that is used to perform a single, related action.

---

### How to Define a Function (Syntax)

You define a function using the `def` keyword, followed by the function name, parentheses `()`, and a colon `:`. The code block for the function is indented.

```python
# 1. Using 'def' keyword
# 2. Function name (e.g., 'greet')
# 3. Parentheses for parameters (we'll cover these later, for now they're empty)
# 4. Colon (:)
# 5. Indented code block (the "recipe steps")

def greet():
    print("Hello there!")
    print("Welcome to Python functions.")

# This is just defining the function. Nothing happens yet!
```

---

### How to Use (Call) a Function

Defining a function doesn't execute its code. To make the function run, you need to "call" it by its name followed by parentheses.

```python
def greet():
    print("Hello there!")
    print("Welcome to Python functions.")

# Now, let's call the function:
greet() # This will execute the code inside the 'greet' function
greet() # You can call it multiple times!
```

**Output:**
```
Hello there!
Welcome to Python functions.
Hello there!
Welcome to Python functions.
```

---

### What is the Purpose of Functions in Programming?

Functions are incredibly important for several reasons:

1.  **Reusability:** This is the primary benefit. You write a piece of code once, and you can use it many times throughout your program (or even in different programs) without rewriting it. This saves time and reduces errors.

2.  **Organization and Modularity:** Functions help you break down complex problems into smaller, manageable pieces. Each function can handle a specific sub-task. This makes your code much easier to understand, develop, and test.

3.  **Readability:** Code organized into functions is easier to read and understand. When you see a function call like `calculate_average()`, you immediately know what that part of the code is trying to achieve, even if you don't look at the function's internal details.

4.  **Maintainability:** If you need to change how a specific task is performed (e.g., updating a calculation), you only need to modify the code in one place (inside the function definition), and those changes will automatically apply everywhere the function is called. This makes your code easier to debug and update.

---

### Simple Example with Parameters

Functions can also accept inputs, called **parameters** (or arguments), to make them more flexible.

```python
def greet_person(name): # 'name' is a parameter
    print(f"Hello, {name}!")
    print("How are you today?")

greet_person("Alice") # Calling the function with "Alice" as the input
greet_person("Bob")   # Calling it again with "Bob"
```

**Output:**
```
Hello, Alice!
How are you today?
Hello, Bob!
How are you today?
```

Here, `greet_person` is a more flexible function because it can greet *any* person whose name you provide.

---

In essence, functions are powerful tools that help you write cleaner, more efficient, and more understandable code!


In [24]:
import gradio as gr

In [35]:
ai_tutor_interface_simple = gr.Interface(
    fn = get_ai_tutor_response,
    inputs = gr.Textbox(lines = 2, placeholder= "Ask the AI Tutor anything...", label= "Your Question"),
    outputs= gr.Textbox(label = "AI Tutor's Answer"),
    title= "🤖 Simple AI Tutor",
    description= "Enter your question below and the AI Tutor will provide an explanation. Powered by GeminiAI.",
    flagging_mode = "never",
)
print("Launching Gradio Interface...")
ai_tutor_interface_simple.launch()

Launching Gradio Interface...
* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [29]:
def stream_ai_tutor_response(user_question):
    """
    Sends a question to the GeminiAI API and streams the response as a generator.
    Args:
        user_question (str): The question asked by the user.
    Yields:
        str: Chunks of the AI's response. 
    """
    system_prompt = "You are a helpful and patient AI Tutor. Explain concepts clearly and concisely."

    try:
        stream = gemini_client.generate_content(
            f"{system_prompt}\n\nStudent question: {user_question}",
            generation_config=genai.types.GenerationConfig(
                temperature=0.7
            ),
            stream=True
        )
        full_response = ""
        for chunk in stream:
            if chunk.text:
                text_chunk = chunk.text
                full_response += text_chunk
                yield full_response
    
    except Exception as e:
        yield f"Sorry, I encountered an error: {e}"


In [36]:
ai_tutor_interface_streaming = gr.Interface(
    fn = stream_ai_tutor_response,
    inputs = gr.Textbox(lines = 2, placeholder= "Ask the AI Tutor anything...", label= "Your Question"),
    outputs= gr.Markdown(
        label = "AI Tutor's Answer (Streaming)", container= True, height= 250
    ),
    title= "🤖 AI Tutor with Streaming",
    description= "Enter your question. The answer will appear word-by-word!",
    flagging_mode = "never",
)
print("Launching Streaming Gradio Interface...")
ai_tutor_interface_streaming.launch()

Launching Streaming Gradio Interface...
* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [31]:
explanation_levels= {
    1: "like I'm 5 years old",
    2: "Like I'm 10 years old",
    3: "like a high school student",
    4: "like a college student",
    5: "like an expert in the field"
}

In [33]:
def stream_ai_tutor_response_with_level(user_question, explanation_level_value):
    """
    Streams AI Tutor response based on user question and selected explanation level.
    Args:
        user_question (str): The question from the user.
        explanation_level_value (int): The value from the slider (1-5).
    
    Yields:
        str: Chunks of the AI's response.
    """
    level_description = explanation_levels.get(
        explanation_level_value, "clearly and concisely"
    )
    system_prompt = f"You are a helpful AI Tutor. Explain the following concept {level_description}"
    print(f"DEBUG: Using System Prompt: '{system_prompt}'")

    try:
        stream = gemini_client.generate_content(
            f"{system_prompt}\n\nStudent question: {user_question}",
            generation_config=genai.types.GenerationConfig(
                temperature=0.7
            ),
            stream=True
        )
        full_response = ""

        for chunk in stream:
            if chunk.text:
                text_chunk = chunk.text
                full_response += text_chunk
                yield full_response
    except Exception as e:
        print(f"An error occurred during streaming: {e}")
        yield f"Sorry, I encountered an error: {e}"

In [ ]:
ai_tutor_interface_slider = gr.Interface(
    fn= stream_ai_tutor_response_with_level,
    inputs=[
        gr.Textbox(lines = 3, placeholder= "Ask the AI Tutor a question...", label = "Your Question"),
        gr.Slider(
            minimum= 1,
            maximum= 5,
            step = 1,
            value= 3, #default level (high school)
            label= "Explanation Level",
        ),
    ],
    outputs= gr.Markdown(label="AI Tutor's Explanation (Streaming)", container= True, height= 250),
    title= "🎓 Advanced AI Tutor",
    description= "Ask a question and select and desired level of explanation using the slider.",
    flagging_mode = "never",
)
print("Launching Advanced Interface with Slider...")
ai_tutor_interface_slider.launch()

Launching Advanced Interface with Slider...
* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


DEBUG: Using System Prompt: 'You are a helpful AI Tutor. Explain the following concept like a high school student'
DEBUG: Using System Prompt: 'You are a helpful AI Tutor. Explain the following concept like a college student'
